In [ ]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().resolve().parent
dreamt_pi = pd.read_csv(ROOT/'data'/'raw'/'dreamt'/'participant_info.csv')
print("Age\n",dreamt_pi.AGE.describe())
print("\nApnea-Hypopnea Index\n",dreamt_pi.AHI.describe())
print("\n",dreamt_pi.GENDER.value_counts())
dreamt_pi.columns


Age
 count    100.00000
mean      56.23550
std       16.55955
min       21.23000
25%       42.73250
50%       58.69500
75%       70.15750
max       86.93000
Name: AGE, dtype: float64

Apnea-Hypopnea Index
 count    100.000000
mean      22.130000
std       28.681693
min        0.000000
25%        4.000000
50%       13.500000
75%       29.250000
max      182.000000
Name: AHI, dtype: float64

 GENDER
F    55
M    45
Name: count, dtype: int64


Index(['SID', 'AGE', 'GENDER', 'BMI', 'OAHI', 'AHI', 'Mean_SaO2',
       'Arousal Index', 'MEDICAL_HISTORY', 'Sleep_Disorders'],
      dtype='str')

In [ ]:
import yaml 
import os
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve().parent
with open(ROOT/'config.yaml','r') as file:
    config = yaml.safe_load(file)

dreamt_data_path = ROOT/config['dreamt']['path']/'data_64Hz'
dreamt_audit = []
dreamt_missing = []
for file in os.listdir(dreamt_data_path):
    df = pd.read_csv(dreamt_data_path/file, usecols=['TIMESTAMP','Sleep_Stage'])
    n = df.shape[0]
    dreamt_audit.append({'participant':file[:4],
                         'duration':df['TIMESTAMP'].iloc[-1] - df['TIMESTAMP'].iloc[0],
                         'n_rows': n,
                         'label_epoch_counts':df['Sleep_Stage'].value_counts()//config['dreamt']['rows_per_epoch']})

    total_missing=0
    for chunk in pd.read_csv(dreamt_data_path/file, usecols=['BVP', 'ACC_X', 'ACC_Y', 'ACC_Z', 'TEMP', 'EDA', 'HR', 'IBI', 'Sleep_Stage', 'Obstructive_Apnea', 'Central_Apnea', 'Hypopnea', 'Multiple_Events'], chunksize=100_000):
        total_missing += chunk.isna().sum()
    missing_fraction = total_missing/n
    dreamt_missing.append({'participant': file[:4],'missing_fraction': missing_fraction})


output_dir = ROOT/'results'/'inventory'
output_dir.mkdir(parents=True, exist_ok=True)

dreamt_missing_df = pd.DataFrame([{'participant': row['participant'],
                                       **{f'missing_{col}': value for col, value in row['missing_fraction'].items()}} for row in dreamt_missing])    
dreamt_missing_df.to_csv(output_dir/'dreamt_missingness_audit.csv', index=False)
dreamt_audit.to_csv(output_dir/'dreamt_label_audit.csv', index=False)

In [1]:
import os
from pathlib import Path
import yaml
import numpy as np
import pandas as pd

# 1. Setup Paths & Load Config
ROOT = Path.cwd().resolve().parent if Path.cwd().name != 'project' else Path.cwd().resolve()
with open(ROOT / 'config.yaml', 'r') as file:
    config = yaml.safe_load(file)

dreamt_data_path = ROOT / config['dreamt']['path'] / 'data_64Hz'
output_dir = ROOT / 'results' / 'inventory'
output_dir.mkdir(parents=True, exist_ok=True)

rows_per_epoch = config['dreamt'].get('rows_per_epoch', 1920)
target_dt = 1.0 / 64.0 

dreamt_sync_audit = []

# 2. Iterate Across DREAMT Participants
for file in sorted(os.listdir(dreamt_data_path)):
    if not file.endswith('.csv'):
        continue
        
    file_path = dreamt_data_path / file
    participant_id = file[:4]

    df = pd.read_csv(file_path, usecols=['TIMESTAMP', 'Sleep_Stage'])
    n_rows = df.shape[0]

    # Handle numeric vs string timestamps for 64Hz sampling check
    if pd.api.types.is_numeric_dtype(df['TIMESTAMP']):
        time_diffs = df['TIMESTAMP'].diff().dropna()
    else:
        timestamps = pd.to_datetime(df['TIMESTAMP'], unit='s', errors='coerce')
        time_diffs = timestamps.diff().dt.total_seconds().dropna()

    if len(time_diffs) > 0:
        median_dt = time_diffs.median()
        is_median_valid = np.isclose(median_dt, target_dt, atol=1e-4)
        is_diffs_valid = (np.abs(time_diffs - target_dt) < 1e-3).mean() > 0.999
        grid_valid = bool(is_median_valid and is_diffs_valid)
    else:
        grid_valid = False

    # Find first active PSG scoring row (transition from preparation 'P')
    non_p_indices = df.index[df['Sleep_Stage'] != 'P'].tolist()
    if not non_p_indices:
        dreamt_sync_audit.append({
            'participant': participant_id,
            'n_rows': n_rows,
            'first_scored_row': None,
            'grid_64hz_valid': grid_valid,
            'label_blocks_aligned': False,
            'status': 'FAIL',
            'reason': 'Only preparation(P) stage found'
        })
        continue

    first_scored_row = non_p_indices[0]

    # Calculate full 1920-row epoch boundaries anchored at first_scored_row
    total_scored_rows = n_rows - first_scored_row
    full_epochs_count = total_scored_rows // rows_per_epoch
    valid_end_index = first_scored_row + (full_epochs_count * rows_per_epoch)

    # Slice scored portion up to the last complete 30-second epoch
    df_full_scored = df.iloc[first_scored_row:valid_end_index].copy()
    df_full_scored['stage_change'] = (df_full_scored['Sleep_Stage'] != df_full_scored['Sleep_Stage'].shift()).cumsum()
    
    # Check if all internal sleep stage transitions fall cleanly on 1920-row bounds
    block_lengths = df_full_scored.groupby('stage_change')['Sleep_Stage'].agg(['first', 'count'])
    misaligned_blocks = block_lengths[block_lengths['count'] % rows_per_epoch != 0]

    label_blocks_aligned = (len(misaligned_blocks) == 0)

    # Status Determination
    if grid_valid and label_blocks_aligned:
        status = 'PASS'
        reason = 'Aligned with 1920-row grid'
    else:
        status = 'FAIL'
        reasons = []
        if not grid_valid:
            reasons.append('Irregular 64Hz sampling grid')
        if len(misaligned_blocks) > 0:
            reasons.append(f'{len(misaligned_blocks)} misaligned block boundaries')
        reason = '; '.join(reasons)

    dreamt_sync_audit.append({
        'participant': participant_id,
        'n_rows': n_rows,
        'first_scored_row': first_scored_row,
        'grid_64hz_valid': grid_valid,
        'label_blocks_aligned': label_blocks_aligned,
        'status': status,
        'reason': reason
    })

# 3. Export Audit Results
df_audit = pd.DataFrame(dreamt_sync_audit)
df_audit.to_csv(output_dir / 'dreamt_sync_audit.csv', index=False)

print(f"Audit completed across {len(df_audit)} DREAMT participants.")
print(f"Pass: {(df_audit['status'] == 'PASS').sum()}\nFail: {(df_audit['status'] == 'FAIL').sum()}")
print(f"Results written to: {output_dir / 'dreamt_sync_audit.csv'}")

Audit completed across 100 DREAMT participants.
Pass: 97
Fail: 3
Results written to: D:\Adobe\projects\research proj\results\inventory\dreamt_sync_audit.csv


In [2]:
import re
from pathlib import Path
import pandas as pd

# 1. Setup Paths
ROOT = Path.cwd().resolve().parent if Path.cwd().name != 'project' else Path.cwd().resolve()
input_csv = ROOT / 'results' / 'inventory' / 'dreamt_label_audit.csv'
output_csv = ROOT / 'results' / 'inventory' / 'dreamt_label_audit.csv'

# 2. Read Raw CSV
df = pd.read_csv(input_csv)

# 3. Parser Function to Extract Stage-Count Pairs via Regex
def parse_stage_counts(text_block):
    if not isinstance(text_block, str):
        return {}
    # Matches stage names (W, N1, N2, N3, R, P, Missing) followed by integer counts
    matches = re.findall(r'(W|N1|N2|N3|R|P|Missing)\s+(\d+)', text_block)
    return {stage: int(count) for stage, count in matches}

# 4. Apply Parser & Expand into Columns
parsed_dicts = df['label_epoch_counts'].apply(parse_stage_counts)
df_counts = pd.DataFrame(parsed_dicts.tolist()).fillna(0).astype(int)

# 5. Rename Columns with Suffix
df_counts.columns = [f"{col}_epoch_counts" for col in df_counts.columns]

# Reorder columns logically if present
expected_stages = ['W_epoch_counts', 'N1_epoch_counts', 'N2_epoch_counts', 
                   'N3_epoch_counts', 'R_epoch_counts', 'P_epoch_counts', 'Missing_epoch_counts']
ordered_cols = [col for col in expected_stages if col in df_counts.columns]
df_counts = df_counts[ordered_cols]

# 6. Merge Back & Save
df_final = pd.concat([df[['participant', 'duration', 'n_rows']], df_counts], axis=1)
df_final.to_csv(output_csv, index=False)

print(f"SUCCESS: Parsed stage counts saved to {output_csv}")
print(df_final.head())

SUCCESS: Parsed stage counts saved to D:\Adobe\projects\research proj\results\inventory\dreamt_label_audit.csv
  participant  duration   n_rows  W_epoch_counts  N1_epoch_counts  \
0        S002   31464.0  2013697             265               64   
1        S003   32064.0  2052097             114               32   
2        S004   30971.0  1982145             248              163   
3        S005   31123.0  1991873              93               39   
4        S006   33411.0  2138305             252              123   

   N2_epoch_counts  N3_epoch_counts  R_epoch_counts  P_epoch_counts  \
0              334                0              80             305   
1              428              136             119             239   
2              424                1               0             196   
3              480                5             116             304   
4              290              178               0             270   

   Missing_epoch_counts  
0                    